## Linear Regression in Statistical view 


We will use these equations to implement our Model :

- $\hat{Y}= \hat{\beta}_0 + \hat{\beta}_1 * x$
- $R^2=1-\frac{SSE}{SST}$
- $Adjusted \enspace R^2 =1 − (1 − R^2)\frac{n−k} {n−1}$ where $k$ : Number of estimators
- $\hat{\beta}_0= \bar{y}-\hat{\beta}_1 \bar{x}$

- for ridge regularization we used :
    $\hat{\beta}_1= \cfrac{Sxy}{Sxx+\lambda}$
  
- for lasso regularization we used :
    $\hat{\beta}_1=\cfrac{Sxy-\cfrac \lambda 2}{Sxx} $

### Importing required libraries
This cell imports numpy, pandas, and plotly for data manipulation and visualization.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


In [2]:
class statisticalRegression: 
    def __init__(self):
        self.beta_0=None 
        self.beta_1=None
        self.mse=None
        self.sse=None
        self.sst=None #Syy
        self.sxx=None
        self.sxy=None
        self.r_squared=None
        self.adjusted_r_squared=None
        self.lmbda=0 
        self.p=2
        
    def fit(self,y,x,lamb ,reg):
        self.lmbda=lamb
        Y = np.array(y)
        X = np.array(x)
        n= X.shape[0]
        x_bar= np.mean(X)
        y_bar= np.mean(Y)
        self.sxx=np.sum(X**2 - n*x_bar**2)
        self.sst=np.sum(Y**2 - n*y_bar**2) #Syy
        self.sxy=np.sum(X*Y)-n*x_bar*y_bar
        if reg == 'lasso' : self.lasso(Y,X)
        else  : self.ridge(Y,X)
        self.beta_0= y_bar - self.beta_1*x_bar
        Y_hat=self.beta_0 +self.beta_1*X
        e=Y-Y_hat
        self.sse=np.sum(e**2)
        self.mse=self.sse/(n-self.p) 
        self.r_squared= 1-(self.sse/self.sst) 
        self.adjusted_r_squared=1-(1-self.r_squared) * (n-1)/(n-self.p)

    def lasso (self,Y,X):
        halfLambda=self.lmbda/2
        if abs(self.sxy)> halfLambda:
            self.beta_1= (np.sign(self.sxy)*(abs(self.sxy)-halfLambda) )/self.sxx
        else:
            self.beta_1=0

    def ridge(self,Y,X):
        self.beta_1=self.sxy/(self.sxx+self.lmbda)
        
    def predict(self,x):
        X = np.array(x)
        predictedY= self.beta_0+self.beta_1*X
        return predictedY
        
    def summary(self):
        print(f"R squared = {self.r_squared}")
        print(f"Adjusted R squared = {self.adjusted_r_squared}")
        print(f"Numer of predictors = {self.p}")


    def plot(self, X , Y ,y_pred):
    
        fig = go.Figure()
        
        # Scatter (data points)
        fig.add_trace(go.Scatter(
            x=X,
            y=Y,
            mode='markers',
            name='Data points'
        ))
        
        # Regression line
        fig.add_trace(go.Scatter(
            x=X,
            y=y_pred,
            mode='lines',
            name='Regression line'
        ))
    
        # Layout
        fig.update_layout(
            title=f"Simple Linear Regression: y = {self.beta_0:.2f} + {self.beta_1:.2f}x",
            xaxis_title="X",
            yaxis_title="y"
        )
    
        fig.show()

### Loading the dataset
This cell loads the processed insurance dataset and extracts training and testing features and labels.

In [3]:
data = np.load('insurance_processed.npz', allow_pickle=True)
X_train = data['X_train']        
Y_train = data['Y_train']        
X_test  = data['X_test']         
Y_test  = data['Y_test']         
feature_names = data['feature_names']

### Extracting the 'smoker' feature
This cell selects the 'smoker' column from the features for use in regression modeling.

In [4]:
smoker_index = list(feature_names).index('smoker')
X_train_smoker = X_train[:, smoker_index]
X_test_smoker  = X_test[:, smoker_index]

### Training Ridge regression on 'smoker' feature
This cell fits a Ridge regression model using the 'smoker' feature and displays the model summary.

In [5]:
smoke_ridge_model = statisticalRegression()
smoke_ridge_model.fit(Y_train, X_train_smoker, lamb=1.0, reg='ridge')
smoke_ridge_model.summary()

R squared = 1.0002912844223175
Adjusted R squared = 1.0002915574161528
Numer of predictors = 2


In [6]:
print("Evaluation of the model")
print("------------------------")
print(f'Line of best fit is: y = {smoke_ridge_model.beta_0:.3f} + {smoke_ridge_model.beta_1:.3f} x')
print(f'Mean squared error is: {smoke_ridge_model.mse:.3f}')

Evaluation of the model
------------------------
Line of best fit is: y = 13325.634 + 9422.288 x
Mean squared error is: 55302829.596


In [7]:
y_pred_smoker_ridge = smoke_ridge_model.predict(X_test_smoker)

In [8]:
smoke_ridge_model.plot(X_test_smoker, Y_test , y_pred_smoker_ridge)

### Training Lasso regression on 'smoker' feature
This cell fits a Lasso regression model using the 'smoker' feature and displays the model summary.

In [9]:
smoke_lasso_model = statisticalRegression()
smoke_lasso_model.fit(Y_train, X_train_smoker, lamb=1.0, reg='lasso')
smoke_lasso_model.summary()

R squared = 1.0002912840123583
Adjusted R squared = 1.0002915570058095
Numer of predictors = 2


In [10]:
print("Evaluation of the model")
print("------------------------")
print(f'Line of best fit is: y = {smoke_lasso_model.beta_0:.3f} + {smoke_lasso_model.beta_1:.3f} x')
print(f'Mean squared error is: {smoke_lasso_model.mse:.3f}')

Evaluation of the model
------------------------
Line of best fit is: y = 13325.634 + 9431.102 x
Mean squared error is: 55302751.762


In [11]:
y_pred_smoker_lasso = smoke_lasso_model.predict(X_test_smoker)

In [12]:
smoke_lasso_model.plot(X_test_smoker, Y_test , y_pred_smoker_lasso)

## ---------------------------------------------------------

### Extracting the 'age' feature
This cell selects the 'age' column from the features for use in regression modeling.

In [13]:
age_index = list(feature_names).index('age')

X_train_age = X_train[:, age_index]
X_test_age  = X_test[:, age_index]

### Training Ridge regression on 'age' feature
This cell fits a Ridge regression model using the 'age' feature and displays the model summary.

In [14]:
age_ridge_model = statisticalRegression()
age_ridge_model.fit(Y_train, X_train_age, lamb=1.0, reg='ridge')
age_ridge_model.summary()

R squared = 1.000693963867622
Adjusted R squared = 1.0006946142555018
Numer of predictors = 2


In [15]:
print("Evaluation of the model")
print("------------------------")
print(f'Line of best fit is: y = {age_ridge_model.beta_0:.3f} + {age_ridge_model.beta_1:.3f} x')
print(f'Mean squared error is: {age_ridge_model.mse:.3f}')

Evaluation of the model
------------------------
Line of best fit is: y = 13325.634 + 3551.469 x
Mean squared error is: 131754953.497


In [16]:
y_pred_age_ridge = age_ridge_model.predict(X_test_age)

age_ridge_model.plot(X_test_age, Y_test , y_pred_age_ridge)

In [17]:
# create figure
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(
    x=X_test_age,
    y=Y_test,
    mode='markers',
    name='Actual (Y_test)',
    marker=dict(size=8)
))

# Predicted values
fig.add_trace(go.Scatter(
    x=X_test_age,
    y=y_pred_age_ridge,
    mode='markers',
    name='Predicted (y_pred)',
    marker=dict(size=8, symbol='x')
))

# layout
fig.update_layout(
    title="Actual vs Predicted (age Feature)",
    xaxis_title="age ",
    yaxis_title="Charges",
)

fig.show()

### Training Lasso regression on 'age' feature
This cell fits a Lasso regression model using the 'age' feature and displays the model summary.

In [18]:
age_lasso_model = statisticalRegression()
age_lasso_model.fit(Y_train, X_train_age, lamb=1.0, reg='lasso')
age_lasso_model.summary()

R squared = 1.000693963809379
Adjusted R squared = 1.0006946141972042
Numer of predictors = 2


In [19]:
print("Evaluation of the model")
print("------------------------")
print(f'Line of best fit is: y = {age_lasso_model.beta_0:.3f} + {age_lasso_model.beta_1:.3f} x')
print(f'Mean squared error is: {age_lasso_model.mse:.3f}')

Evaluation of the model
------------------------
Line of best fit is: y = 13325.634 + 3554.791 x
Mean squared error is: 131754942.439


In [20]:
y_pred_age_lasso = age_lasso_model.predict(X_test_age)

age_lasso_model.plot(X_test_age, Y_test , y_pred_age_lasso)

In [21]:
# create figure
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(
    x=X_test_age,
    y=Y_test,
    mode='markers',
    name='Actual (Y_test)',
    marker=dict(size=8)
))

# Predicted values
fig.add_trace(go.Scatter(
    x=X_test_age,
    y=y_pred_age_lasso,
    mode='markers',
    name='Predicted (y_pred)',
    marker=dict(size=8, symbol='x')
))

# layout
fig.update_layout(
    title="Actual vs Predicted (age Feature)",
    xaxis_title="age ",
    yaxis_title="Charges",
)

fig.show()